# EDA + Feature Engineering + Baseline Modeling

This notebook provides a comprehensive workflow for:
1. **Exploratory Data Analysis (EDA)**
2. **Feature Engineering**
3. **Baseline Model Development**

## Setup and Installation

In [ ]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Machine Learning
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder, MinMaxScaler, RobustScaler
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.feature_selection import SelectKBest, f_regression, f_classif, RFE
from sklearn.impute import SimpleImputer, KNNImputer

# Advanced models
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor, CatBoostClassifier

# Statistical analysis
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# EDA tools
import missingno as msno

# Settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

# Ignore warnings
import warnings
warnings.filterwarnings('ignore')

print("Libraries imported successfully!")

## 1. Data Loading and Initial Exploration

In [ ]:
# Load your dataset
# Replace 'your_dataset.csv' with your actual file path
# df = pd.read_csv('your_dataset.csv')
# For Excel files: df = pd.read_excel('your_dataset.xlsx')
# For JSON files: df = pd.read_json('your_dataset.json')

# For demonstration, let's create a sample dataset
# Remove this and load your actual data
np.random.seed(42)
sample_size = 1000

data = {
    'age': np.random.randint(18, 80, sample_size),
    'income': np.random.normal(50000, 15000, sample_size),
    'education_years': np.random.randint(8, 20, sample_size),
    'experience_years': np.random.randint(0, 40, sample_size),
    'gender': np.random.choice(['Male', 'Female', 'Other'], sample_size),
    'city': np.random.choice(['New York', 'Los Angeles', 'Chicago', 'Houston', 'Phoenix'], sample_size),
    'satisfaction_score': np.random.randint(1, 11, sample_size),
    'target': np.random.choice([0, 1], sample_size, p=[0.6, 0.4])
}

df = pd.DataFrame(data)

# Add some missing values for demonstration
missing_indices = np.random.choice(df.index, size=int(0.05*len(df)), replace=False)
df.loc[missing_indices, 'income'] = np.nan
missing_indices = np.random.choice(df.index, size=int(0.03*len(df)), replace=False)
df.loc[missing_indices, 'education_years'] = np.nan

print(f"Dataset shape: {df.shape}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
# Basic information about the dataset
print("Dataset Info:")
df.info()

print("\n" + "="*50)
print("Statistical Summary:")
df.describe().T

In [ ]:
# Check for missing values
print("Missing Values:")
missing_data = df.isnull().sum()
missing_percentage = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
})
print(missing_df[missing_df['Missing Count'] > 0])

# Visualize missing values
plt.figure(figsize=(12, 6))
msno.matrix(df)
plt.title('Missing Values Matrix')
plt.show()

## 2. Comprehensive EDA

In [ ]:
# Separate numerical and categorical columns
numerical_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical columns: {numerical_cols}")
print(f"Categorical columns: {categorical_cols}")

# Remove target variable from features if it exists
target_col = 'target' if 'target' in df.columns else None
if target_col:
    numerical_cols.remove(target_col)

In [ ]:
# Univariate Analysis - Numerical Variables
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.ravel()

for i, col in enumerate(numerical_cols[:6]):
    if i < len(axes):
        # Histogram
        axes[i].hist(df[col].dropna(), bins=30, alpha=0.7, edgecolor='black')
        axes[i].set_title(f'Distribution of {col}')
        axes[i].set_xlabel(col)
        axes[i].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Box plots for numerical variables
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.ravel()

for i, col in enumerate(numerical_cols[:6]):
    if i < len(axes):
        axes[i].boxplot(df[col].dropna())
        axes[i].set_title(f'Box Plot of {col}')
        axes[i].set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
# Categorical Variables Analysis
if categorical_cols:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()
    
    for i, col in enumerate(categorical_cols[:4]):
        if i < len(axes):
            value_counts = df[col].value_counts()
            axes[i].bar(value_counts.index, value_counts.values)
            axes[i].set_title(f'Distribution of {col}')
            axes[i].set_xlabel(col)
            axes[i].set_ylabel('Count')
            axes[i].tick_params(axis='x', rotation=45)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation Analysis
if len(numerical_cols) > 1:
    # Pearson correlation
    correlation_matrix = df[numerical_cols].corr()
    
    plt.figure(figsize=(12, 8))
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', center=0,
                square=True, linewidths=0.5)
    plt.title('Correlation Matrix')
    plt.show()

## 3. Feature Engineering

In [ ]:
# Create a copy for feature engineering
df_fe = df.copy()

# 1. Handling Missing Values
print("Handling missing values...")

# Numerical columns - use median for robustness
numerical_imputer = SimpleImputer(strategy='median')
df_fe[numerical_cols] = numerical_imputer.fit_transform(df_fe[numerical_cols])

# Categorical columns - use mode
if categorical_cols:
    categorical_imputer = SimpleImputer(strategy='most_frequent')
    df_fe[categorical_cols] = categorical_imputer.fit_transform(df_fe[categorical_cols])

print("Missing values handled!")
print(f"Remaining missing values: {df_fe.isnull().sum().sum()}")

In [ ]:
# 2. Outlier Detection and Treatment
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

print("Outlier Analysis:")
for col in numerical_cols:
    outliers, lower, upper = detect_outliers_iqr(df_fe, col)
    print(f"{col}: {len(outliers)} outliers detected ({len(outliers)/len(df_fe)*100:.2f}%)")

In [ ]:
# 3. Feature Creation
print("Creating new features...")

# Example feature engineering (customize based on your data)
if 'age' in df_fe.columns and 'experience_years' in df_fe.columns:
    df_fe['age_experience_ratio'] = df_fe['experience_years'] / (df_fe['age'] + 1)
    df_fe['career_start_age'] = df_fe['age'] - df_fe['experience_years']

if 'income' in df_fe.columns and 'education_years' in df_fe.columns:
    df_fe['income_per_education_year'] = df_fe['income'] / df_fe['education_years']

if 'satisfaction_score' in df_fe.columns:
    df_fe['satisfaction_category'] = pd.cut(df_fe['satisfaction_score'], 
                                          bins=[0, 3, 7, 10], 
                                          labels=['Low', 'Medium', 'High'])

print(f"Original features: {df.shape[1]}")
print(f"After feature engineering: {df_fe.shape[1]}")

In [ ]:
# 4. Encoding Categorical Variables
print("Encoding categorical variables...")

# Update categorical columns after feature creation
categorical_cols = df_fe.select_dtypes(include=['object', 'category']).columns.tolist()

# Label Encoding for ordinal variables
label_encoders = {}
for col in categorical_cols:
    if df_fe[col].nunique() <= 10:  # For low cardinality
        le = LabelEncoder()
        df_fe[f'{col}_encoded'] = le.fit_transform(df_fe[col])
        label_encoders[col] = le
        print(f"Label encoded {col}")

# One-Hot Encoding for nominal variables
high_cardinality_cols = [col for col in categorical_cols if df_fe[col].nunique() > 10]
if high_cardinality_cols:
    df_fe = pd.get_dummies(df_fe, columns=high_cardinality_cols, prefix=high_cardinality_cols)
    print(f"One-hot encoded: {high_cardinality_cols}")

print(f"Final dataset shape after encoding: {df_fe.shape}")

In [ ]:
# 5. Feature Scaling
print("Scaling features...")

# Update numerical columns
numerical_cols = df_fe.select_dtypes(include=[np.number]).columns.tolist()
if target_col:
    numerical_cols = [col for col in numerical_cols if col != target_col]

# StandardScaler for most features
scaler = StandardScaler()
df_fe[numerical_cols] = scaler.fit_transform(df_fe[numerical_cols])

print("Features scaled using StandardScaler")
print(f"Scaled {len(numerical_cols)} numerical features")

## 4. Baseline Model Development

In [ ]:
# Prepare data for modeling
if target_col:
    X = df_fe.drop(columns=[target_col])
    y = df_fe[target_col]
    
    # Split the data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y if y.nunique() <= 10 else None
    )
    
    print(f"Training set shape: {X_train.shape}")
    print(f"Test set shape: {X_test.shape}")
    print(f"Target distribution: {y.value_counts().to_dict()}")
else:
    print("No target column found. Please specify your target variable.")

In [ ]:
# Define models based on problem type
if target_col:
    is_regression = y.nunique() > 10
    
    if is_regression:
        models = {
            'Linear Regression': LinearRegression(),
            'Ridge Regression': Ridge(alpha=1.0),
            'Lasso Regression': Lasso(alpha=1.0),
            'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42),
            'XGBoost': xgb.XGBRegressor(random_state=42),
            'LightGBM': lgb.LGBMRegressor(random_state=42, verbose=-1),
            'CatBoost': CatBoostRegressor(random_state=42, verbose=False)
        }
        
        metrics = {
            'RMSE': lambda y_true, y_pred: np.sqrt(mean_squared_error(y_true, y_pred)),
            'MAE': mean_absolute_error,
            'R2': r2_score
        }
    else:
        models = {
            'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
            'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
            'XGBoost': xgb.XGBClassifier(random_state=42),
            'LightGBM': lgb.LGBMClassifier(random_state=42, verbose=-1),
            'CatBoost': CatBoostClassifier(random_state=42, verbose=False)
        }
        
        metrics = {
            'Accuracy': accuracy_score,
            'F1': lambda y_true, y_pred: classification_report(y_true, y_pred, output_dict=True)['weighted avg']['f1-score']
        }
    
    print(f"Problem type: {'Regression' if is_regression else 'Classification'}")
    print(f"Models to evaluate: {list(models.keys())}")

In [ ]:
# Train and evaluate baseline models
if target_col:
    results = {}
    
    for name, model in models.items():
        print(f"\nTraining {name}...")
        
        # Train the model
        model.fit(X_train, y_train)
        
        # Make predictions
        y_pred = model.predict(X_test)
        
        # Calculate metrics
        model_results = {}
        for metric_name, metric_func in metrics.items():
            try:
                score = metric_func(y_test, y_pred)
                model_results[metric_name] = score
            except Exception as e:
                print(f"Could not calculate {metric_name}: {e}")
        
        results[name] = model_results
        
        # Cross-validation
        cv_scores = cross_val_score(model, X, y, cv=5, scoring='neg_mean_squared_error' if is_regression else 'accuracy')
        if is_regression:
            results[name]['CV_RMSE'] = np.sqrt(-cv_scores.mean())
        else:
            results[name]['CV_Accuracy'] = cv_scores.mean()
        
        print(f"Results for {name}: {model_results}")
    
    # Display results
    results_df = pd.DataFrame(results).T
    print("\n" + "="*50)
    print("MODEL COMPARISON:")
    print(results_df.round(4))

In [ ]:
# Visualize model performance
if target_col:
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))
    axes = axes.ravel()
    
    # Plot 1: Main metric comparison
    main_metric = 'R2' if is_regression else 'Accuracy'
    results_df[main_metric].sort_values().plot(kind='barh', ax=axes[0])
    axes[0].set_title(f'Model Comparison - {main_metric}')
    axes[0].set_xlabel(main_metric)
    
    # Plot 2: CV performance
    cv_metric = 'CV_RMSE' if is_regression else 'CV_Accuracy'
    results_df[cv_metric].sort_values().plot(kind='barh', ax=axes[1])
    axes[1].set_title(f'Model Comparison - {cv_metric}')
    axes[1].set_xlabel(cv_metric)
    
    # Plot 3: Feature importance (for tree-based models)
    best_model_name = results_df[main_metric].idxmax()
    best_model = models[best_model_name]
    
    if hasattr(best_model, 'feature_importances_'):
        importances = best_model.feature_importances_
        feature_names = X.columns
        
        # Get top 10 features
        indices = np.argsort(importances)[::-1][:10]
        
        axes[2].barh(range(len(indices)), importances[indices])
        axes[2].set_yticks(range(len(indices)))
        axes[2].set_yticklabels([feature_names[i] for i in indices])
        axes[2].set_title(f'Top 10 Feature Importance - {best_model_name}')
        axes[2].set_xlabel('Importance')
    else:
        axes[2].text(0.5, 0.5, 'Feature importance not available', 
                    ha='center', va='center', transform=axes[2].transAxes)
    
    # Plot 4: Predictions vs Actual (for regression)
    if is_regression:
        y_pred_best = best_model.predict(X_test)
        axes[3].scatter(y_test, y_pred_best, alpha=0.6)
        axes[3].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
        axes[3].set_xlabel('Actual')
        axes[3].set_ylabel('Predicted')
        axes[3].set_title(f'Predictions vs Actual - {best_model_name}')
    else:
        # Confusion matrix for classification
        y_pred_best = best_model.predict(X_test)
        cm = confusion_matrix(y_test, y_pred_best)
        sns.heatmap(cm, annot=True, fmt='d', ax=axes[3], cmap='Blues')
        axes[3].set_title(f'Confusion Matrix - {best_model_name}')
        axes[3].set_xlabel('Predicted')
        axes[3].set_ylabel('Actual')
    
    plt.tight_layout()
    plt.show()

## 5. Summary and Next Steps

In [ ]:
# Final summary
print("\n" + "="*80)
print("EDA + FEATURE ENGINEERING + BASELINE MODELING - SUMMARY")
print("="*80)

print(f"\n📊 Dataset Overview:")
print(f"   • Original shape: {df.shape}")
print(f"   • Final shape after processing: {df_fe.shape}")
print(f"   • Missing values handled: {df.isnull().sum().sum()}")
print(f"   • Features created: {df_fe.shape[1] - df.shape[1]}")

if target_col:
    print(f"\n🤖 Modeling Results:")
    print(f"   • Problem type: {'Regression' if is_regression else 'Classification'}")
    print(f"   • Models evaluated: {len(models)}")
    print(f"   • Best model: {best_model_name}")
    print(f"   • Best {main_metric}: {results_df.loc[best_model_name, main_metric]:.4f}")

print(f"\n🚀 Next Steps:")
print(f"   1. Load your actual dataset and replace the sample data")
print(f"   2. Customize feature engineering based on your domain knowledge")
print(f"   3. Perform hyperparameter tuning on the best model")
print(f"   4. Try ensemble methods and advanced techniques")
print(f"   5. Validate model on unseen data")
print(f"   6. Deploy the best model for production use")

print("\n" + "="*80)